In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from itertools import combinations
import os
from scipy import sparse

In [2]:
cd ./adatas/RRErythroidC

d:\Projects\Geneformer\adatas\RRErythroidC


d:\Gaol\miniconda3\envs\geneformer_env\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
ls

 Volume in drive D has no label.
 Volume Serial Number is 4EE5-FFF2

 Directory of d:\Projects\Geneformer\adatas\RRErythroidC

02/08/2026  12:43 AM    <DIR>          .
02/08/2026  12:32 AM    <DIR>          ..
02/02/2026  03:41 PM       147,974,898 RR_HC_NK.h5ad
02/08/2026  12:07 AM       180,014,598 RR_HC_PolychromaticEE.h5ad
02/08/2026  12:06 AM       375,487,946 RR_HC_RBCLE.h5ad
02/08/2026  12:19 AM       136,169,454 RR_post_nres_BasophilicAML.h5ad
02/08/2026  12:19 AM       119,147,914 RR_post_nres_BasophilicLE.h5ad
02/08/2026  12:19 AM       126,300,338 RR_post_nres_macrophageEE.h5ad
02/08/2026  12:19 AM       142,388,834 RR_post_nres_PolychromaticLE.h5ad
02/08/2026  12:19 AM       365,678,294 RR_post_nres_RBCLE.h5ad
02/08/2026  12:19 AM       136,874,386 RR_post_nres_unknownLE.h5ad
02/08/2026  12:24 AM       152,871,694 RR_post_res_PolychromaticAML.h5ad
02/08/2026  12:24 AM       161,882,210 RR_post_res_PolychromaticLE.h5ad
02/08/2026  12:24 AM       228,587,374 RR_post_res_RBCLE

In [ ]:
AML系
pre_res_PolychromaticAMLvsHC_NK
pre_res_PolychromaticAMLvsRR_post_nres_BasophilicAML
pre_res_PolychromaticAMLvsHC_post_res_PolychromaticAML
pre_res_PolychromaticAMLvsHC_pre_nres_DCAM

LE系
pre_res_PolychromaticLEvsHC_RBCLE
pre_res_PolychromaticLEvsRR_post_nres_PolychromaticLE
pre_res_PolychromaticLEvsRR_post_nres_BasophilicLE
pre_res_PolychromaticLEvsRR_post_nres_unknownLE
pre_res_PolychromaticLEvsRR_post_res_RBCLE
pre_res_PolychromaticLEvsRR_pre_nres_PolychromaticLE
pre_res_PolychromaticLEvsRR_pre_nres_RBCLE
pre_res_PolychromaticLEvsRR_pre_res_PolychromaticLE

HC_RBCLEvsRR_post_nres_PolychromaticLE
HC_RBCLEvsRR_post_nres_BasophilicLE
HC_RBCLEvsRR_post_nres_unknownLE
HC_RBCLEvsRR_post_res_RBCLE
HC_RBCLEvsRR_pre_nres_PolychromaticLE
HC_RBCLEvsRR_pre_nres_RBCLE
HC_RBCLEvsRR_pre_res_PolychromaticLE


EE系
HC_PolychromaticEEvspost_nres_macrophageEE
HC_PolychromaticEEvspre_nres_RBCEE
HC_PolychromaticEEvspre_res_PolychromaticEE
pre_res_PolychromaticEEvspost_nres_macrophageEE
pre_res_PolychromaticEEvspre_nres_RBCEE

In [9]:
clusters_to_test = [
    # ================= AML系 =================
    ("RR_pre_res_PolychromaticAMLvsRR_HC_NK",
     "RR_pre_res_PolychromaticAML", "RR_HC_NK"),

    ("RR_pre_res_PolychromaticAMLvsRR_post_nres_BasophilicAML",
     "RR_pre_res_PolychromaticAML", "RR_post_nres_BasophilicAML"),

    # 你原文写了 "HC_post_res_PolychromaticAML"：文件里实际是 RR_post_res_PolychromaticAML
    ("RR_pre_res_PolychromaticAMLvsRR_post_res_PolychromaticAML",
     "RR_pre_res_PolychromaticAML", "RR_post_res_PolychromaticAML"),

    # 你原文写了 "HC_pre_nres_DCAM"：文件里实际是 RR_pre_nres_DCAML
    ("RR_pre_res_PolychromaticAMLvsRR_pre_nres_DCAML",
     "RR_pre_res_PolychromaticAML", "RR_pre_nres_DCAML"),


    # ================= LE系：pre_res_PolychromaticLE 为 g1 =================
    ("RR_pre_res_PolychromaticLEvsRR_HC_RBCLE",
     "RR_pre_res_PolychromaticLE", "RR_HC_RBCLE"),

    ("RR_pre_res_PolychromaticLEvsRR_post_nres_PolychromaticLE",
     "RR_pre_res_PolychromaticLE", "RR_post_nres_PolychromaticLE"),

    ("RR_pre_res_PolychromaticLEvsRR_post_nres_BasophilicLE",
     "RR_pre_res_PolychromaticLE", "RR_post_nres_BasophilicLE"),

    ("RR_pre_res_PolychromaticLEvsRR_post_nres_unknownLE",
     "RR_pre_res_PolychromaticLE", "RR_post_nres_unknownLE"),

    ("RR_pre_res_PolychromaticLEvsRR_post_res_RBCLE",
     "RR_pre_res_PolychromaticLE", "RR_post_res_RBCLE"),

    ("RR_pre_res_PolychromaticLEvsRR_pre_nres_PolychromaticLE",
     "RR_pre_res_PolychromaticLE", "RR_pre_nres_PolychromaticLE"),

    ("RR_pre_res_PolychromaticLEvsRR_pre_nres_RBCLE",
     "RR_pre_res_PolychromaticLE", "RR_pre_nres_RBCLE"),

    ("RR_pre_res_PolychromaticLEvsRR_pre_res_PolychromaticLE",
     "RR_pre_res_PolychromaticLE", "RR_pre_res_PolychromaticLE"),


    # ================= LE系：HC_RBCLE 为 g1 =================
    ("RR_HC_RBCLEvsRR_post_nres_PolychromaticLE",
     "RR_HC_RBCLE", "RR_post_nres_PolychromaticLE"),

    ("RR_HC_RBCLEvsRR_post_nres_BasophilicLE",
     "RR_HC_RBCLE", "RR_post_nres_BasophilicLE"),

    ("RR_HC_RBCLEvsRR_post_nres_unknownLE",
     "RR_HC_RBCLE", "RR_post_nres_unknownLE"),

    ("RR_HC_RBCLEvsRR_post_res_RBCLE",
     "RR_HC_RBCLE", "RR_post_res_RBCLE"),

    ("RR_HC_RBCLEvsRR_pre_nres_PolychromaticLE",
     "RR_HC_RBCLE", "RR_pre_nres_PolychromaticLE"),

    ("RR_HC_RBCLEvsRR_pre_nres_RBCLE",
     "RR_HC_RBCLE", "RR_pre_nres_RBCLE"),

    ("RR_HC_RBCLEvsRR_pre_res_PolychromaticLE",
     "RR_HC_RBCLE", "RR_pre_res_PolychromaticLE"),


    # ================= EE系 =================
    ("RR_HC_PolychromaticEEvsRR_post_nres_macrophageEE",
     "RR_HC_PolychromaticEE", "RR_post_nres_macrophageEE"),

    ("RR_HC_PolychromaticEEvsRR_pre_nres_RBCEE",
     "RR_HC_PolychromaticEE", "RR_pre_nres_RBCEE"),

    ("RR_HC_PolychromaticEEvsRR_pre_res_PolychromaticEE",
     "RR_HC_PolychromaticEE", "RR_pre_res_PolychromaticEE"),

    ("RR_pre_res_PolychromaticEEvsRR_post_nres_macrophageEE",
     "RR_pre_res_PolychromaticEE", "RR_post_nres_macrophageEE"),

    ("RR_pre_res_PolychromaticEEvsRR_pre_nres_RBCEE",
     "RR_pre_res_PolychromaticEE", "RR_pre_nres_RBCEE"),
]


In [17]:

# =========================================================
# 1) 参数（与你原逻辑一致；仅把 FC 设为 1.5）
# =========================================================
USE_LAYER = None        # e.g. "counts" / "lognorm"；不用就 None
USE_RAW = False         # 若要用 adata.raw，就 True（优先于 layers）
THRESHOLD = 0.0         # positive-only：只用 >threshold 的表达细胞
MIN_POS_CELLS = 20      # 每组至少多少表达细胞才纳入
USE_FDR = True          # 建议 True：多重检验
P_CUTOFF = 0.05
FDR_CUTOFF = 0.05

FC_CUTOFF = 1.3        # ✅ 你要的：FC > 1.5
FC_METHOD = "mean"      # "mean" 或 "median"
FC_PSEUDOCOUNT = 0.0    # 如担心除零，可设 1e-9

# 是否先做 normalize/log1p（强烈建议 True，除非你的 X 已经是 lognorm）
DO_PREPROCESS = False
TARGET_SUM = 1e4

# 输出目录
OUTDIR = "DE_by_cluster"
os.makedirs(OUTDIR, exist_ok=True)

DATA_DIR = r"d:\Projects\Geneformer\adatas\RRErythroidC"

group_files = {
    os.path.splitext(f)[0]: os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.lower().endswith(".h5ad")
}

# =========================================================
# 2) 工具函数
# =========================================================
def bh_fdr(pvals):
    """Benjamini–Hochberg FDR；返回与输入同长度的 qvals（np.array）"""
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    if ok.sum() == 0:
        return qvals
    pv = pvals[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    qvals[ok] = out
    return qvals

def get_matrix_and_genes(adata, use_layer=None, use_raw=False):
    """
    返回 (X, gene_names)
    X: cells x genes (csr_matrix or ndarray)
    gene_names: np.array of gene symbols
    """
    if use_raw:
        if adata.raw is None:
            return None, None
        X = adata.raw.X
        genes = np.asarray(adata.raw.var_names)
    elif use_layer is not None:
        if use_layer not in adata.layers:
            return None, None
        X = adata.layers[use_layer]
        genes = np.asarray(adata.var_names)
    else:
        X = adata.X
        genes = np.asarray(adata.var_names)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = np.asarray(X)
    return X, genes

def summary_stat(x, method="mean"):
    if method == "median":
        return float(np.median(x))
    return float(np.mean(x))

def conditional_mw_positive_only_allgenes_okonly(
    ad_res, ad_nres, res_name, nres_name,
    min_pos_cells=20, threshold=0.0,
    use_layer=None, use_raw=False,
    fc_method="mean",
    fc_pseudocount=0.0
):
    """
    对所有共有基因做 positive-only MWU：
      - 仅使用 >threshold 的细胞
      - 若任一组 positive cells < min_pos_cells：丢弃该基因
      - 计算 FC（nres/res）和 log2FC（nres/res），基于 positive-only 且用 mean/median
    """
    X1, genes1 = get_matrix_and_genes(ad_res, use_layer=use_layer, use_raw=use_raw)
    X2, genes2 = get_matrix_and_genes(ad_nres, use_layer=use_layer, use_raw=use_raw)
    if X1 is None or X2 is None:
        return pd.DataFrame()

    common = np.intersect1d(genes1, genes2, assume_unique=False)
    if common.size == 0:
        return pd.DataFrame()

    idx1 = pd.Index(genes1).get_indexer(common)
    idx2 = pd.Index(genes2).get_indexer(common)

    X1c = X1[:, idx1]
    X2c = X2[:, idx2]

    rows = []
    for j, gene in enumerate(common):
        if sparse.issparse(X1c):
            x1_all = X1c[:, j].toarray().ravel()
            x2_all = X2c[:, j].toarray().ravel()
        else:
            x1_all = np.asarray(X1c[:, j]).ravel()
            x2_all = np.asarray(X2c[:, j]).ravel()

        x1_pos = x1_all[x1_all > threshold]  # g1
        x2_pos = x2_all[x2_all > threshold]  # g2

        if (x1_pos.size < min_pos_cells) or (x2_pos.size < min_pos_cells):
            continue

        u, p = mannwhitneyu(x1_pos, x2_pos, alternative="two-sided")

        stat_g1 = summary_stat(x1_pos, method=fc_method)
        stat_g2 = summary_stat(x2_pos, method=fc_method)

        denom = stat_g1 + fc_pseudocount
        numer = stat_g2 + fc_pseudocount
        fc = (numer / denom) if denom > 0 else np.nan
        log2fc = np.log2(fc) if (np.isfinite(fc) and fc > 0) else np.nan

        rows.append({
            "gene": gene,
            "p_value": float(p),
            "U": float(u),
            f"{res_name}_n_pos": int(x1_pos.size),
            f"{nres_name}_n_pos": int(x2_pos.size),
            f"{fc_method}_g1_pos": float(stat_g1),
            f"{fc_method}_g2_pos": float(stat_g2),
            "FC_g2_over_g1": float(fc) if np.isfinite(fc) else np.nan,
            "log2FC_g2_over_g1": float(log2fc) if np.isfinite(log2fc) else np.nan,
            "mean_pos_diff(g1-g2)": float(np.mean(x1_pos) - np.mean(x2_pos)),
        })

    return pd.DataFrame(rows)

def maybe_preprocess(adata: sc.AnnData) -> sc.AnnData:
    """
    可选：对每个 adata 做 normalize_total + log1p（在 copy 上做）
    若你的 X 已经是 lognorm，就把 DO_PREPROCESS 设为 False
    """
    if not DO_PREPROCESS:
        return adata
    ad = adata.copy()
    sc.pp.normalize_total(ad, target_sum=TARGET_SUM)
    sc.pp.log1p(ad)
    return ad

# =========================================================
# 2.5) 检查：pairs 用到的 key 都存在
# =========================================================
missing = []
for name, g1, g2 in clusters_to_test:
    if g1 not in group_files:
        missing.append(g1)
    if g2 not in group_files:
        missing.append(g2)

if missing:
    raise KeyError(
        "Missing keys in group_files (check spelling / file names):\n"
        + "\n".join(sorted(set(missing)))
        + "\n\nAvailable keys example:\n"
        + "\n".join(list(sorted(group_files.keys()))[:10])
    )

# =========================================================
# 3) 读取数据
# =========================================================
adatas = {}
for k, fp in group_files.items():
    # 只读本次要用到的 key（可选：省内存）
    # 这里不改变结构太多：直接全部读也行，但更占内存
    pass

# 仅加载本次 pairs 需要的文件，避免一次性把 20 个都读进内存
needed_keys = sorted(set([g1 for _, g1, _ in clusters_to_test] + [g2 for _, _, g2 in clusters_to_test]))
for k in needed_keys:
    fp = group_files[k]
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing file for {k}: {fp}")
    adatas[k] = sc.read_h5ad(fp)

# =========================================================
# 4) 主流程：逐 pair 跑 DE（g2 相对 g1 上调）
# =========================================================
all_sig_tables = []
log2fc_by_cluster = {}   # pair -> pd.Series(gene -> log2FC)
sig_gene_sets = {}       # pair -> set(sig_up_genes)

for pair_name, g1_key, g2_key in clusters_to_test:
    ad_g1 = maybe_preprocess(adatas[g1_key])
    ad_g2 = maybe_preprocess(adatas[g2_key])

    df = conditional_mw_positive_only_allgenes_okonly(
        ad_g1, ad_g2,
        res_name=g1_key, nres_name=g2_key,
        min_pos_cells=MIN_POS_CELLS,
        threshold=THRESHOLD,
        use_layer=USE_LAYER,
        use_raw=USE_RAW,
        fc_method=FC_METHOD,
        fc_pseudocount=FC_PSEUDOCOUNT
    )

    if df.shape[0] == 0:
        print(f"[WARN] {pair_name}: no genes passed MIN_POS_CELLS.")
        continue

    df["FDR_BH"] = bh_fdr(df["p_value"].values)

    # 显著性 + 上调(g2>g1) + FC>1.5
    if USE_FDR:
        sig_mask = (
            df["FDR_BH"].notna()
            & (df["FDR_BH"] < FDR_CUTOFF)
            & df["log2FC_g2_over_g1"].notna()
            & (df["log2FC_g2_over_g1"] > np.log2(FC_CUTOFF))
        )
    else:
        sig_mask = (
            df["p_value"].notna()
            & (df["p_value"] < P_CUTOFF)
            & df["log2FC_g2_over_g1"].notna()
            & (df["log2FC_g2_over_g1"] > np.log2(FC_CUTOFF))
        )

    up_mask = df["log2FC_g2_over_g1"].notna() & (df["log2FC_g2_over_g1"] > 0)
    sig_up = df[sig_mask & up_mask].copy()

    # 排序：更靠前=更显著
    sig_up = sig_up.sort_values(["FDR_BH", "p_value"], ascending=True)

    # 保存
    out_full = os.path.join(OUTDIR, f"{pair_name}__full_all_ok_genes.csv")
    out_sig  = os.path.join(OUTDIR, f"{pair_name}__SIG_up_in_{g2_key}__FCgt{FC_CUTOFF}.csv")
    df.to_csv(out_full, index=False)
    sig_up.to_csv(out_sig, index=False)

    print(f"[{pair_name}] ok_genes={df.shape[0]}  sig_up={sig_up.shape[0]}")
    print(f"  saved: {out_sig}")

    # 汇总用
    sig_up["pair"] = pair_name
    all_sig_tables.append(sig_up)

    log2fc_by_cluster[pair_name] = df.set_index("gene")["log2FC_g2_over_g1"]
    sig_gene_sets[pair_name] = set(sig_up["gene"].tolist())

# =========================================================
# 5) 合并输出：所有 pair 的显著上调基因清单
# =========================================================
combined_path = os.path.join(OUTDIR, f"ALL_pairs__SIG_up_combined__FCgt{FC_CUTOFF}.csv")
perpair_list_path = os.path.join(OUTDIR, f"ALL_pairs__SIG_up_gene_list_per_pair__FCgt{FC_CUTOFF}.csv")

if len(all_sig_tables) > 0:
    all_sig_df = pd.concat(all_sig_tables, ignore_index=True)
    all_sig_df.to_csv(combined_path, index=False)

    # 列表式：每个 pair 一行
    list_rows = []
    for pair_name, genes in sig_gene_sets.items():
        list_rows.append({
            "pair": pair_name,
            "n_sig_up_genes": len(genes),
            "genes_sig_up_in_g2": ";".join(sorted(list(genes)))
        })
    pd.DataFrame(list_rows).to_csv(perpair_list_path, index=False)

    print("\n[OK] Combined saved:")
    print(" ", combined_path)
    print(" ", perpair_list_path)
else:
    print("\n[WARN] No significant genes found in any pair. Nothing to combine.")

# =========================================================
# 6) （可选）把 combined 结果展开成 wide 表：gene × pair
#    这部分保持你原思路，但把路径和列名对齐
# =========================================================
# 你如果不需要 wide 输出，可以把下面整段注释掉

MAKE_WIDE = True

if MAKE_WIDE and os.path.exists(combined_path):
    df = pd.read_csv(combined_path)

    # ===== 只保留你关心的 pair（这里默认全保留；要筛就改 pairs_keep）=====
    # pairs_keep = [ ... ]  # 例如只留 AML 系等
    # df = df[df["pair"].isin(pairs_keep)]

    # ===== 设置多级索引：gene × pair =====
    df_idx = df.set_index(["gene", "pair"])

    # ===== 选择要展开的指标列 =====
    value_cols = [
        "p_value",
        # "FDR_BH",
        "FC_g2_over_g1",
        # "log2FC_g2_over_g1",
        f"{FC_METHOD}_g1_pos",
        f"{FC_METHOD}_g2_pos",
    ]
    value_cols = [c for c in value_cols if c in df.columns]

    # ===== unstack pair → 自动补 NaN =====
    df_wide = df_idx[value_cols].unstack("pair")

    # ===== 调整列顺序：pair 在前，metric 在后 =====
    df_wide = df_wide.swaplevel(0, 1, axis=1)
    df_wide = df_wide.sort_index(axis=1, level=0)

    # ===== 压平成单层列名 =====
    df_wide.columns = [f"{pair}__{metric}" for pair, metric in df_wide.columns]

    wide_path = os.path.join(OUTDIR, f"SIG_up_wide__FCgt{FC_CUTOFF}.csv")
    df_wide.to_csv(wide_path)
    print("\n[OK] Wide table saved:")
    print(" ", wide_path)


[RR_pre_res_PolychromaticAMLvsRR_HC_NK] ok_genes=1169  sig_up=145
  saved: DE_by_cluster\RR_pre_res_PolychromaticAMLvsRR_HC_NK__SIG_up_in_RR_HC_NK__FCgt1.3.csv
[RR_pre_res_PolychromaticAMLvsRR_post_nres_BasophilicAML] ok_genes=2798  sig_up=28
  saved: DE_by_cluster\RR_pre_res_PolychromaticAMLvsRR_post_nres_BasophilicAML__SIG_up_in_RR_post_nres_BasophilicAML__FCgt1.3.csv
[RR_pre_res_PolychromaticAMLvsRR_post_res_PolychromaticAML] ok_genes=2820  sig_up=5
  saved: DE_by_cluster\RR_pre_res_PolychromaticAMLvsRR_post_res_PolychromaticAML__SIG_up_in_RR_post_res_PolychromaticAML__FCgt1.3.csv
[RR_pre_res_PolychromaticAMLvsRR_pre_nres_DCAML] ok_genes=2858  sig_up=183
  saved: DE_by_cluster\RR_pre_res_PolychromaticAMLvsRR_pre_nres_DCAML__SIG_up_in_RR_pre_nres_DCAML__FCgt1.3.csv
[RR_pre_res_PolychromaticLEvsRR_HC_RBCLE] ok_genes=3688  sig_up=113
  saved: DE_by_cluster\RR_pre_res_PolychromaticLEvsRR_HC_RBCLE__SIG_up_in_RR_HC_RBCLE__FCgt1.3.csv
[RR_pre_res_PolychromaticLEvsRR_post_nres_Polychromatic

In [16]:

if MAKE_WIDE and os.path.exists(combined_path):
    df = pd.read_csv(combined_path)

    # ===== 只保留你关心的 pair（这里默认全保留；要筛就改 pairs_keep）=====
    # pairs_keep = [ ... ]  # 例如只留 AML 系等
    # df = df[df["pair"].isin(pairs_keep)]

    # ===== 设置多级索引：gene × pair =====
    df_idx = df.set_index(["gene", "pair"])

    # ===== 选择要展开的指标列 =====
    value_cols = [
        "p_value",
        # "FDR_BH",
        "FC_g2_over_g1",
        # "log2FC_g2_over_g1",
        f"{FC_METHOD}_g1_pos",
        f"{FC_METHOD}_g2_pos",
    ]
    value_cols = [c for c in value_cols if c in df.columns]

    # ===== unstack pair → 自动补 NaN =====
    df_wide = df_idx[value_cols].unstack("pair")

    # ===== 调整列顺序：pair 在前，metric 在后 =====
    df_wide = df_wide.swaplevel(0, 1, axis=1)
    df_wide = df_wide.sort_index(axis=1, level=0)

    # ===== 压平成单层列名 =====
    df_wide.columns = [f"{pair}__{metric}" for pair, metric in df_wide.columns]

    wide_path = os.path.join(OUTDIR, f"SIG_up_wide__FCgt{FC_CUTOFF}.csv")
    df_wide.to_csv(wide_path)
    print("\n[OK] Wide table saved:")
    print(" ", wide_path)


[OK] Wide table saved:
  DE_by_cluster\SIG_up_wide__FCgt1.csv


In [ ]:



group_files = {
    # ================= pre =================
    "HC_Erythroid": os.path.join(DATA_DIR, "RR_HC_Erythroid.h5ad"),
    "post_nres_Erythroid": os.path.join(DATA_DIR, "RR_post_nres_Erythroid.h5ad"),
    "pre_res_Erythroid": os.path.join(DATA_DIR, "RR_pre_res_Erythroid.h5ad"),
    "pre_nres_Erythroid": os.path.join(DATA_DIR, "RR_pre_nres_Erythroid.h5ad"),

}
clusters_to_test = [
    ("HCvspost_nres",  "HC_Erythroid",  "post_nres_Erythroid"),
    ("HCvspre_nres",  "HC_Erythroid",  "pre_nres_Erythroid"),
    ("HCvspre_res",  "HC_Erythroid",  "pre_res_Erythroid"),
]

# =========================================================
# 1) 参数（跟你之前逻辑一致）
# =========================================================
USE_LAYER = None       # e.g. "counts" / "lognorm"；不用就 None
USE_RAW = False        # 若要用 adata.raw，就 True（优先于 layers）
THRESHOLD = 0.0       # positive-only：只用 >threshold 的表达细胞
MIN_POS_CELLS = 20     # 每组至少多少表达细胞才纳入
USE_FDR = True         # 建议 True：多重检验
P_CUTOFF = 0.05
FDR_CUTOFF = 0.05
FC_CUTOFF = 1
FC_METHOD = "mean"     # "mean" 或 "median"
FC_PSEUDOCOUNT = 0.0   # 如担心除零，可设 1e-9

# 是否先做 normalize/log1p（强烈建议 True，除非你的 X 已经是 lognorm）
DO_PREPROCESS = False
TARGET_SUM = 1e4

# 输出目录
OUTDIR = "DE_by_cluster"
os.makedirs(OUTDIR, exist_ok=True)

# =========================================================
# 2) 工具函数
# =========================================================
def bh_fdr(pvals):
    """Benjamini–Hochberg FDR；返回与输入同长度的 qvals（np.array）"""
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    if ok.sum() == 0:
        return qvals
    pv = pvals[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    qvals[ok] = out
    return qvals

def get_matrix_and_genes(adata, use_layer=None, use_raw=False):
    """
    返回 (X, gene_names)
    X: cells x genes (csr_matrix or ndarray)
    gene_names: np.array of gene symbols
    """
    if use_raw:
        if adata.raw is None:
            return None, None
        X = adata.raw.X
        genes = np.asarray(adata.raw.var_names)
    elif use_layer is not None:
        if use_layer not in adata.layers:
            return None, None
        X = adata.layers[use_layer]
        genes = np.asarray(adata.var_names)
    else:
        X = adata.X
        genes = np.asarray(adata.var_names)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = np.asarray(X)
    return X, genes

def summary_stat(x, method="mean"):
    if method == "median":
        return float(np.median(x))
    return float(np.mean(x))

def conditional_mw_positive_only_allgenes_okonly(
    ad_res, ad_nres, res_name, nres_name,
    min_pos_cells=20, threshold=0.0,
    use_layer=None, use_raw=False,
    fc_method="mean",
    fc_pseudocount=0.0
):
    """
    对所有共有基因做 positive-only MWU：
      - 仅使用 >threshold 的细胞
      - 若任一组 positive cells < min_pos_cells：丢弃该基因
      - 计算 FC（nres/res）和 log2FC（nres/res），基于 positive-only 且用 mean/median
    """
    X1, genes1 = get_matrix_and_genes(ad_res, use_layer=use_layer, use_raw=use_raw)
    X2, genes2 = get_matrix_and_genes(ad_nres, use_layer=use_layer, use_raw=use_raw)
    if X1 is None or X2 is None:
        return pd.DataFrame()

    common = np.intersect1d(genes1, genes2, assume_unique=False)
    if common.size == 0:
        return pd.DataFrame()

    idx1 = pd.Index(genes1).get_indexer(common)
    idx2 = pd.Index(genes2).get_indexer(common)

    X1c = X1[:, idx1]
    X2c = X2[:, idx2]

    rows = []
    for j, gene in enumerate(common):
        if sparse.issparse(X1c):
            x1_all = X1c[:, j].toarray().ravel()
            x2_all = X2c[:, j].toarray().ravel()
        else:
            x1_all = np.asarray(X1c[:, j]).ravel()
            x2_all = np.asarray(X2c[:, j]).ravel()

        x1_pos = x1_all[x1_all > threshold]  # res
        x2_pos = x2_all[x2_all > threshold]  # nres

        if (x1_pos.size < min_pos_cells) or (x2_pos.size < min_pos_cells):
            continue

        u, p = mannwhitneyu(x1_pos, x2_pos, alternative="two-sided")

        stat_res  = summary_stat(x1_pos, method=fc_method)
        stat_nres = summary_stat(x2_pos, method=fc_method)

        denom = stat_res + fc_pseudocount
        numer = stat_nres + fc_pseudocount
        fc = (numer / denom) if denom > 0 else np.nan
        log2fc = np.log2(fc) if (np.isfinite(fc) and fc > 0) else np.nan

        rows.append({
            "gene": gene,
            "p_value": float(p),
            "U": float(u),
            f"{res_name}_n_pos": int(x1_pos.size),
            f"{nres_name}_n_pos": int(x2_pos.size),
            f"{fc_method}_g1_pos": float(stat_res),
            f"{fc_method}_g2_pos": float(stat_nres),
            f"FC_g2_over_g1": float(fc) if np.isfinite(fc) else np.nan,
            f"log2FC_g2_over_g1": float(log2fc) if np.isfinite(log2fc) else np.nan,
            # f"median_pos_diff({nres_name}-{res_name})": float(np.median(x1_pos) - np.median(x2_pos)),
            f"mean_pos_diff(g1-g2)": float(np.mean(x1_pos) - np.mean(x2_pos)),
        })

    return pd.DataFrame(rows)

def maybe_preprocess(adata: sc.AnnData) -> sc.AnnData:
    """
    可选：对每个 adata 做 normalize_total + log1p（在 copy 上做）
    若你的 X 已经是 lognorm（比如 Scanpy 标准流程），就把 DO_PREPROCESS 设为 False
    """
    if not DO_PREPROCESS:
        return adata
    ad = adata.copy()
    sc.pp.normalize_total(ad, target_sum=TARGET_SUM)
    sc.pp.log1p(ad)
    return ad

# =========================================================
# 3) 读取数据
# =========================================================
adatas = {}
for k, fp in group_files.items():
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing file for {k}: {fp}")
    adatas[k] = sc.read_h5ad(fp)

# =========================================================
# 4) 主流程：逐 cluster 跑 DE（pre_nres vs pre_res）
# =========================================================
all_sig_tables = []
log2fc_by_cluster = {}   # cluster -> pd.Series(gene -> log2FC)
sig_gene_sets = {}       # cluster -> set(sig_up_genes)

for cluster_name, res_key, nres_key in clusters_to_test:
    ad_res = maybe_preprocess(adatas[res_key])
    ad_nres = maybe_preprocess(adatas[nres_key])

    df = conditional_mw_positive_only_allgenes_okonly(
        ad_res, ad_nres,
        res_name=res_key, nres_name=nres_key,
        min_pos_cells=MIN_POS_CELLS,
        threshold=THRESHOLD,
        use_layer=USE_LAYER,
        use_raw=USE_RAW,
        fc_method=FC_METHOD,
        fc_pseudocount=FC_PSEUDOCOUNT
    )

    if df.shape[0] == 0:
        print(f"[WARN] {cluster_name}: no genes passed MIN_POS_CELLS.")
        continue

    df["FDR_BH"] = bh_fdr(df["p_value"].values)

    # 显著性 + 上调(nres>res)
    if USE_FDR:
        sig_mask = df["FDR_BH"].notna() & (df["FDR_BH"] < FDR_CUTOFF) & (df[f"log2FC_g2_over_g1"] > np.log2(FC_CUTOFF))
    else:
        sig_mask = df["p_value"].notna() & (df["p_value"] < P_CUTOFF) & (df[f"log2FC_g2_over_g1"] > np.log2(FC_CUTOFF))

    up_mask = df[f"log2FC_g2_over_g1"].notna() & (df[f"log2FC_g2_over_g1"] > 0)
    sig_up = df[sig_mask & up_mask].copy()

    # 排序：更靠前=更显著
    sig_up = sig_up.sort_values(["FDR_BH", "p_value"], ascending=True)

    # 保存
    out_full = os.path.join(OUTDIR, f"{cluster_name}__full_all_ok_genes.csv")
    out_sig  = os.path.join(OUTDIR, f"{cluster_name}__SIG_up_in_{nres_key}.csv")
    df.to_csv(out_full, index=False)
    sig_up.to_csv(out_sig, index=False)

    print(f"[{cluster_name}] ok_genes={df.shape[0]}  sig_up={sig_up.shape[0]}")
    print(f"  saved: {out_sig}")

    # 汇总用
    sig_up["cluster"] = cluster_name
    all_sig_tables.append(sig_up)

    log2fc_by_cluster[cluster_name] = df.set_index("gene")[f"log2FC_g2_over_g1"]
    sig_gene_sets[cluster_name] = set(sig_up["gene"].tolist())

# 合并输出：所有 cluster 的显著上调基因清单（你要的 “comprehensively list”）
if len(all_sig_tables) > 0:
    all_sig_df = pd.concat(all_sig_tables, ignore_index=True)
    all_sig_df.to_csv(os.path.join(OUTDIR, f"ALL_clusters__SIG_up_combined.csv"), index=False)

    # 也给一个更“列表式”的版本：每个 cluster 一行，genes 用分号拼起来
    list_rows = []
    for cluster_name, genes in sig_gene_sets.items():
        list_rows.append({
            "cluster": cluster_name,
            "n_sig_up_genes": len(genes),
            "genes_sig_up_in_pre_nres": ";".join(sorted(list(genes)))
        })
    pd.DataFrame(list_rows).to_csv(
        os.path.join(OUTDIR, f"ALL_clusters__SIG_up_gene_list_per_cluster.csv"),
        index=False
    )




# ===== 1. 读入数据 =====
# df = pd.read_csv("../../Gao/RR/Upregulated in Post-nonresponder compared to pre-nonresponder0.csv")
df = pd.read_csv("./DE_pre_nres_vs_pre_res_by_cluster/ALL_clusters__SIG_up_in_pre_res_Erythroid__combined.csv")

# 如果列名和你实际的不完全一致，可以在这里统一
df = df.rename(columns={
    "Gene": "gene",          # 如果原来是 Gene
    "cluster": "pair",      # 如果原来不是 pair
})

# ===== 2. 只保留你关心的 pair（可选，但推荐）=====
pairs_keep = ["HCvspre_res", "HCvspre_nres", 'HCvspost_nres']
df = df[df["pair"].isin(pairs_keep)]

# ===== 3. 设置多级索引：gene × pair =====
df_idx = df.set_index(["gene", "pair"])

# ===== 4. 选择要展开的指标列 =====
# value_cols = [
#     "p_value",
#     "mean_pre_pos",
#     "mean_post_pos",
#     "FC_post_over_pre"
# ]
value_cols = [
    "p_value",
    "mean_g1_pos",
    "mean_g2_pos",
    "FC_g2_over_g1"
]
# ===== 5. unstack pair → 自动补 NaN =====
df_wide = (
    df_idx[value_cols]
    .unstack("pair")        # columns 变成 (metric, pair)
)

# ===== 6. 调整列顺序：pair 在前，metric 在后 =====
df_wide = df_wide.swaplevel(0, 1, axis=1)
df_wide = df_wide.sort_index(axis=1, level=0)

# ===== 7. 压平成单层列名（推荐）=====
df_wide.columns = [
    f"{pair}__{metric}"
    for pair, metric in df_wide.columns
]
df_wide.to_csv(f'Upregulated in AML Erythroid compared to HC Erythroid FC{FC_CUTOFF}.csv')